### Python packages used in this code

In [ ]:
import importlib
import sys
utils_path = './'
data_path = './data'
if utils_path not in sys.path:
    sys.path.append(utils_path)

from MyPackages import *
from MyFunctions import *


%matplotlib inline
MySEED = 373



def fun_Reload(pkg):
    module = importlib.import_module(pkg)
    module = importlib.reload(module)
    attrs = {attr: getattr(module, attr) for attr in dir(module)}
    globals().update(attrs)
    
fun_Reload('MyPackages')
fun_Reload('MyFunctions')



print('--Platform--')
print('OS :', platform.platform())
print('--Version--')
print('python : ', sys.version)
print('numpy :', np.__version__)
print('pandas :', pd.__version__)
print('sklearn :', sklearn.__version__)

--Platform--
OS : Linux-6.5.11-linuxkit-x86_64-with-glibc2.31
--Version--
python :  3.9.7 | packaged by conda-forge | (default, Sep 29 2021, 19:23:11) 
[GCC 9.4.0]
numpy : 1.26.4
pandas : 2.0.3
sklearn : 1.3.0


## Environments
--Platform--

OS : Linux-6.5.11-linuxkit-x86_64-with-glibc2.31

--Version--

python :  3.9.7 | packaged by conda-forge | (default, Sep 29 2021, 19:23:11) 

[GCC 9.4.0]

numpy : 1.26.4

pandas : 2.0.3

sklearn : 2.0.3

# Make descriptors
# 1. Load and arrange

In [ ]:
## Descroptor parameter
mp = 10 # parallel number
n = 10 # cyclic oligomerization for polymer
nk = 20 # number of Gaussian in discretization (Hyperparameter)
sigma = 1/nk/2 # band width of Gaussian kernel (Hyperparameter)
mu = None # list of discretizing points (Hyperparameter)
ff_desc = FF_descriptor(GAFF2_mod(), polar =True)

## Source data
source_data = pd.read_csv('./data/PolyOmics/PolyOmics_data.csv', index_col=0).iloc[:,1:]
source_output = source_data.iloc[:,1:]
source_input = ff_desc.ffkm_mp(source_data['SMILES'].to_list(), mp =mp, nk =nk, s=sigma, mu =mu, cyclic=n)

argname_list = source_output.columns.to_list()
print(f'--- Source ---')
print(f'Shape : {source_input.shape, source_output.shape}')

## PoLyInfo data
#PI_data = pd.read_csv('./data/PoLyInfo/PI_data.csv', index_col=0)
#PI_output = PI_data.iloc[:,:3]
#PI_input = ff_desc.ffkm_mp(PI_data['SMILES'].to_list(), mp =mp, nk =nk, s=sigma, mu =mu, cyclic=n)

#print(f'--- PoLyInfo ---')
#print(f'Shape : {PI_input.shape, PI_output.shape}')

## Open data
Open_data = pd.read_csv('./data/Open/Open_data.csv', index_col=0)
Open_output = Open_data.iloc[:,:5]
Open_input = ff_desc.ffkm_mp(Open_output['SMILES'].to_list(), mp =mp, nk =nk, s=sigma, mu =mu, cyclic=n)
# Add LC (Liquid Crystal or not) information 
Open_output['LC'] = (Open_output['N'] + Open_output['S']).where(Open_output['N'] + Open_output['S'] > 2, 1).astype(bool)

print(f'--- Open ---')
print(f'Shape : {Open_input.shape, Open_output.shape}')

--- Source ---
Shape : ((71069, 190), (71069, 30))
--- Open ---
Shape : ((457, 190), (457, 6))


# 2. Scaling
Module `MyScaler()` is defined in `MyFunctions.py`
- `MyScaler(['A', 'B'])`: Define scaler object that scales the data in the order 'A', 'B'.
- `MyScaler.fit_transform(df)` : Estimate scaling parameters and return scaled data.

<div style="float: left">

|**Data**| Scaler |  
|---|---|  
|**Descriptor**|$\log_e$ $\rightarrow$ Normalization $\rightarrow$ Yeo-Johnson|  
|**Properties**|Normalization $\rightarrow$ Yeo-Johnson|  

</div>

## Source data

In [3]:
## Output
dict_outputs = {}
Scalers = {}

for i, argname in enumerate(source_output.columns):

    tmp_data = source_output.copy()
    tmp_column = tmp_data[argname].dropna()
    # Remove outliers
    tmp_column = tmp_column[(tmp_column < tmp_column.quantile(0.95)) &  (tmp_column > tmp_column.quantile(0.05))]
    tmp_Scaler = MyScaler(['Normalization', 'Yeo-Johnson'])
    # tmp_Scaler = MyScaler(['Normalization'])
    tmp_scal = tmp_Scaler.fit_transform(tmp_column)

    dict_outputs[argname] = tmp_scal
    Scalers[argname] = tmp_Scaler
    
    print(f'--- {argname} ---')
    print(f'Shape : {tmp_scal.shape}')
    
    
    
## Inputs
Scaler_inputs = MyScaler(['PlusLog', 'Normalization', 'Yeo-Johnson'])
df_inputs_scal = Scaler_inputs.fit_transform(source_input)

--- density ---
Shape : (63961,)
--- Rg ---
Shape : (63961,)
--- Scaled Rg ---
Shape : (63961,)
--- r2 ---
Shape : (63947,)
--- self-diffusion ---
Shape : (63956,)
--- Cp ---
Shape : (63957,)
--- Cv ---
Shape : (63957,)
--- compressibility ---
Shape : (63957,)
--- bulk_modulus ---
Shape : (63957,)
--- isentropic_compressibility ---
Shape : (63957,)
--- isentropic_bulk_modulus ---
Shape : (63957,)
--- volume_expansion ---
Shape : (63957,)
--- linear_expansion ---
Shape : (63957,)
--- static_dielectric_const ---
Shape : (63961,)
--- dielectric_const_dc ---
Shape : (63134,)
--- nematic_order_parameter ---
Shape : (63961,)
--- refractive_index ---
Shape : (63134,)
--- tg ---
Shape : (39062,)
--- tg_thermal_expansion_coef(upper_tg) ---
Shape : (39062,)
--- tg_thermal_expansion_coef(below_tg) ---
Shape : (39062,)
--- thermal_conductivity ---
Shape : (59227,)
--- thermal_diffusivity ---
Shape : (59224,)
--- TC_ke ---
Shape : (59227,)
--- TC_pe ---
Shape : (59227,)
--- TC_pair ---
Shape : (592

## Target data

In [4]:
#x_target = pd.concat([PI_input, Open_input])
#y_Tm_target = pd.concat([PI_output, Open_output])['Tm'].dropna()
#y_Tiso_target = pd.concat([PI_output, Open_output])['Tiso'].dropna()

# Add negative data of PoLyInfo to NSC classification data
#y_NSC = pd.concat([pd.DataFrame(np.zeros((len(PI_output['LC'].to_frame().query('LC == 0').index),3)), columns=['N', 'S', 'C'], index= PI_output['LC'].to_frame().query('LC == 0').index).astype(bool), Open_output[['N', 'S', 'C']]])
#y_LC = pd.concat([PI_output['LC'], Open_output['LC']]).dropna()
#IMD_idx = pd.concat([PI_data, Open_data]).query('polymer_class == 13').index.to_list()

x_target = Open_input
y_Tm_target = Open_output['Tm'].dropna()
y_Tiso_target = Open_output['Tiso'].dropna()
y_NSC = Open_output[['N', 'S', 'C']]
y_LC = Open_output['LC']
IMD_idx = Open_data.query('polymer_class == 13').index.to_list()

Scaler_target_x = MyScaler(['PlusLog', 'Normalization', 'Yeo-Johnson'])
Scaler_target_y_Tm = MyScaler(['Normalization', 'Yeo-Johnson'])
Scaler_target_y_Tiso = MyScaler(['Normalization', 'Yeo-Johnson'])

x_target_scal = Scaler_target_x.fit_transform(x_target)
y_Tm_target_scal = Scaler_target_y_Tm.fit_transform(y_Tm_target)
y_Tiso_target_scal = Scaler_target_y_Tiso.fit_transform(y_Tiso_target)


print('--- Tm ---')
print(f'Outputs: {y_Tm_target.shape}')
print('--- Tiso ---')
print(f'Outputs: {y_Tiso_target.shape}')

print('--- NSC ---')
print(y_NSC.apply(pd.value_counts))
print('--- LC ---')
print(y_LC.value_counts())

dict_save_target = {
    'Orginal_inputs' : x_target,
    'Orginal_outputs_Tm' : y_Tm_target,
    'Orginal_outputs_Tiso' : y_Tiso_target,
    'IMD_idx' : IMD_idx,
    'Inputs' : x_target_scal,
    'Outputs_Tm' : y_Tm_target_scal,
    'Outputs_Tiso' : y_Tiso_target_scal,
    'Outputs_NSC' : y_NSC,
    'Outputs_LC' : y_LC,
    'Scaler_inputs' : Scaler_target_x,
    'Scaler_outputs_Tm' : Scaler_target_y_Tm,
    'Scaler_outputs_Tiso' : Scaler_target_y_Tiso
}


--- Tm ---
Outputs: (116,)
--- Tiso ---
Outputs: (134,)
--- NSC ---
         N    S    C
False  363  218  122
True    94  239  335
--- LC ---
LC
True    457
Name: count, dtype: int64


# 3. Save

In [5]:
dict_save_source = {
    'Orginal_inputs' : source_input,
    'Orginal_outputs' : source_output,
    'Inputs' : df_inputs_scal,
    'Outputs' : dict_outputs,
    'Scaler_inputs' : Scaler_inputs,
    'Scaler_outputs' : Scalers
}


with open(f'{data_path}/Dataset_source.pkl', 'wb') as f:
    pickle.dump(dict_save_source,f)
f.close()
    
with open(f'{data_path}/Dataset_target.pkl', 'wb') as f:
    pickle.dump(dict_save_target,f)
f.close()